# SEC MAIS — Visão Geral da Camada Bronze

Este notebook realiza a varredura e validação de todos os arquivos **Parquet** gerados pelas extrações na camada **Bronze** (`armazenamento/bronze/`).

### Programas Monitorados:
- 🎓 **Pé-de-Meia** (`pe_de_meia`)
- 🏛️ **Universidade Para Todos** (`universidade_para_todos`)
- 🎒 **Bolsa Presença** (`bolsa_presenca`)
- 📚 **Mais Estudos** (`mais_estudos`)

In [1]:
from pathlib import Path
import os
import pandas as pd
import pyarrow.parquet as pq

# Detecção flexível do diretório da camada bronze
def obter_caminho_bronze() -> Path:
    candidatos = [
        Path.cwd() / "sec-mais" / "armazenamento" / "bronze",
        Path.cwd().parent / "sec-mais" / "armazenamento" / "bronze",
        Path.cwd() / "armazenamento" / "bronze",
        Path.cwd().parent / "armazenamento" / "bronze",
    ]
    for c in candidatos:
        if c.exists():
            return c.resolve()
    return (Path.cwd() / "sec-mais" / "armazenamento" / "bronze").resolve()

caminho_bronze = obter_caminho_bronze()
print(f"📁 Diretório Bronze identificado: {caminho_bronze}")
print(f"   Existe: {caminho_bronze.exists()}")

📁 Diretório Bronze identificado: C:\Users\leonardo.fernandez\Desktop\FERNANDEZ\SEC MAIS\sec-mais\armazenamento\bronze
   Existe: True


## 1. Inventário de Arquivos e Partições na Bronze

In [2]:
def listar_arquivos_bronze(diretorio_base: Path) -> pd.DataFrame:
    registros = []
    if not diretorio_base.exists():
        return pd.DataFrame()

    for arquivo in diretorio_base.glob("**/*.parquet"):
        tamanho_bytes = arquivo.stat().st_size
        modificacao = pd.to_datetime(arquivo.stat().st_mtime, unit="s")
        
        # Extrai programa e partição a partir do caminho relativo
        relativo = arquivo.relative_to(diretorio_base)
        partes = relativo.parts
        programa = partes[0] if len(partes) > 0 else "desconhecido"
        
        # Leitura rápida de metadados sem carregar todos os dados na memória
        try:
            meta = pq.read_metadata(arquivo)
            num_linhas = meta.num_rows
            num_colunas = meta.num_columns
        except Exception as e:
            num_linhas = None
            num_colunas = None

        registros.append({
            "programa": programa,
            "arquivo": arquivo.name,
            "linhas": num_linhas,
            "colunas": num_colunas,
            "tamanho_kb": round(tamanho_bytes / 1024, 2),
            "modificado_em": modificacao,
            "caminho_completo": str(arquivo)
        })

    return pd.DataFrame(registros)

df_inventario = listar_arquivos_bronze(caminho_bronze)

if df_inventario.empty:
    print("⚠️ Nenhum arquivo .parquet encontrado na camada bronze ainda.")
else:
    display(df_inventario[["programa", "arquivo", "linhas", "colunas", "tamanho_kb", "modificado_em"]])

,programa,arquivo,linhas,colunas,tamanho_kb,modificado_em
0,pe_de_meia,pe_de_meia_20261111_235421.parquet,1,5,3.58,2026-11-12 02:54:23.333943129
1,universidade_para_todos,upt_20261111_235417.parquet,1,5,3.65,2026-11-12 02:54:19.338575125


## 2. Resumo por Programa

In [3]:
if not df_inventario.empty:
    resumo = df_inventario.groupby("programa").agg(
        total_arquivos=("arquivo", "count"),
        total_linhas=("linhas", "sum"),
        tamanho_total_kb=("tamanho_kb", "sum"),
        ultima_modificacao=("modificado_em", "max")
    ).reset_index()
    display(resumo)
else:
    print("Sem dados para resumir.")

,programa,total_arquivos,total_linhas,tamanho_total_kb,ultima_modificacao
0,pe_de_meia,1,1,3.58,2026-11-12 02:54:23.333943129
1,universidade_para_todos,1,1,3.65,2026-11-12 02:54:19.338575125


## 3. Pré-visualização dos Dados de Cada Programa

In [4]:
if not df_inventario.empty:
    programas = df_inventario["programa"].unique()
    for prog in programas:
        sub = df_inventario[df_inventario["programa"] == prog]
        ultimo_arquivo = sub.sort_values("modificado_em", ascending=False).iloc[0]["caminho_completo"]
        print(f"\n{'='*70}")
        print(f"🔍 Amostra do programa: {prog.upper()}")
        print(f"📄 Arquivo mais recente: {Path(ultimo_arquivo).name}")
        print(f"{'='*70}")
        
        df_amostra = pd.read_parquet(ultimo_arquivo)
        print(f"Dimensões: {df_amostra.shape[0]} linhas x {df_amostra.shape[1]} colunas")
        print(f"Colunas: {list(df_amostra.columns)}")
        display(df_amostra.head(5))
else:
    print("Nenhum arquivo disponível para pré-visualização.")


🔍 Amostra do programa: PE_DE_MEIA
📄 Arquivo mais recente: pe_de_meia_20261111_235421.parquet
Dimensões: 1 linhas x 5 colunas
Colunas: ['cd_beneficiario', 'nm_beneficiario', 'dt_referencia', '_data_extracao', '_programa']


,cd_beneficiario,nm_beneficiario,dt_referencia,_data_extracao,_programa
0,1,BENEFICIARIO EXEMPLO,2026-11-11,2026-11-11 23:54:21.845772,pe_de_meia



🔍 Amostra do programa: UNIVERSIDADE_PARA_TODOS
📄 Arquivo mais recente: upt_20261111_235417.parquet
Dimensões: 1 linhas x 5 colunas
Colunas: ['cd_beneficiario', 'nm_beneficiario', 'dt_referencia', '_data_extracao', '_programa']


,cd_beneficiario,nm_beneficiario,dt_referencia,_data_extracao,_programa
0,1,BENEFICIARIO EXEMPLO,2026-11-11,2026-11-11 23:54:17.468639,universidade_para_todos
